# Goods suggestion by Collaborative Filtering System with Word2Vec  Chapter 16 Workshop 7

## Install package

In [ ]:
# %pip install openpyxl

## Load module

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec


## Define parameter

In [2]:

import os
cols = 'a:d,g'
# cols = ['product_id', 'category', 'about_product', 'user_id', 'review_content']
data_path = "../datasets/Online_Retail/Online_Retail.xlsx"
if os.path.exists(data_path):
    # data_path = "../datasets/Online_Retail/Online_Retail.csv"
    df = pd.read_excel(data_path, usecols=cols,
                   dtype={'CustomerID': str, 'InvoiceNo': str})
else:
    print(f"File {data_path} does not exist. Please check the path.")
    df = pd.DataFrame()

In [3]:
# df.sample(5, random_state=42)
df.tail()

,InvoiceNo,StockCode,Description,Quantity,CustomerID
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12680
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12680
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12680
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12680
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,12680


## Check data

In [4]:
# df[df['Quantity'] <= 0].sample(5, random_state=42)
# df[df['CustomerID'].isnull()].sample(5, random_state=42)
print(f'Negative quantity: {len(df[df["Quantity"] <= 0])}')
print(df[df["Quantity"] <= 0].count())
print("----------------------------")
print(f'Missing CustomerID: {df[df["CustomerID"].isnull()].count()}')

Negative quantity: 10624
InvoiceNo      10624
StockCode      10624
Description     9762
Quantity       10624
CustomerID      8905
dtype: int64
----------------------------
Missing CustomerID: InvoiceNo      135080
StockCode      135080
Description    133626
Quantity       135080
CustomerID          0
dtype: int64


In [5]:
df.groupby('Quantity').size()

Quantity
-80995    1
-74215    1
-9600     2
-9360     1
-9058     1
         ..
 4800     1
 5568     1
 12540    1
 74215    1
 80995    1
Length: 722, dtype: int64

## Filter unuseful data

### Select data these Quantity more 0

In [6]:
df = df[df['Quantity'] > 0]

In [7]:
df.shape

(531285, 5)

### Convert all column be string type

In [14]:
df['CustomerID'] = df['CustomerID'].astype(str)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

In [27]:
df.sample(5, random_state=42)
# df[:5]

,InvoiceNo,StockCode,Description,Quantity,CustomerID
37804,539482,21557,SET OF 6 FUNKY BEAKERS,1,nan
110208,545681,22149,FELTCRAFT 6 FLOWER FRIENDS,3,16923
112211,545874,21458,2 PICTURE BOOK EGGS EASTER BUNNY,12,12841
41686,539955,48187,DOORMAT NEW ENGLAND,1,nan
515307,579777,20963,APPLE BATH SPONGE,2,nan


In [16]:
# df['CustomerID'] = df['CustomerID'].astype(str)
print(type(df['CustomerID'][0]))
print(type(df['InvoiceNo'][0]))
print(type(df['StockCode'][0]))

<class 'str'>
<class 'str'>
<class 'str'>


### Check invoice that have only 1 quantity

In [30]:
df_invol = df.groupby(['InvoiceNo']).count().Quantity.sort_values()
df_invol

InvoiceNo
A563187       1
561392        1
561386        1
542145        1
561372        1
           ... 
558475      705
580729      721
581492      731
581219      749
573585     1114
Name: Quantity, Length: 20728, dtype: int64

### Find InvoiceNo that less than 2

In [37]:
frame = {'InvoiceNo': df_invol.index, 'Count': df_invol.values}
res = pd.DataFrame(frame)
invoices_small = res[res.Count <= 1].InvoiceNo.tolist()
invoices_small[:10]

['A563187',
 '561392',
 '561386',
 '542145',
 '561372',
 '561368',
 '561365',
 '561361',
 '561333',
 '561327']

In [ ]:
# ~ means not (invert boolean)
df = df[~df['InvoiceNo'].isin(invoices_small)]


In [54]:
print(df.shape)
print(df.groupby('InvoiceNo').size().sort_values().head(10))

(528911, 5)
InvoiceNo
563208    2
565206    2
565205    2
542889    2
554952    2
542871    2
565142    2
565090    2
554944    2
575027    2
dtype: int64


## Prepare dataset

### Create InvoiceNo list

In [59]:
invoiceNoes_ls = df.InvoiceNo.unique().tolist()

In [60]:
print(f"Number of unique InvoiceNoes: {len(invoiceNoes_ls)}")
print(invoiceNoes_ls[:10])

Number of unique InvoiceNoes: 18354
['536365', '536366', '536367', '536368', '536370', '536372', '536373', '536375', '536376', '536377']


### create train and validate dataset

#### shuffle data

In [61]:
import random

print(invoiceNoes_ls[:10])
random.shuffle(invoiceNoes_ls)
print(invoiceNoes_ls[:10])

['536365', '536366', '536367', '536368', '536370', '536372', '536373', '536375', '536376', '536377']
['579980', '541932', '555325', '554512', '576603', '562466', '557787', '574432', '577825', '547364']


#### define train and validate dataset

In [64]:
invoiceNoes_train = invoiceNoes_ls[:int(len(invoiceNoes_ls) * 0.9)]
train_df = df[df['InvoiceNo'].isin(invoiceNoes_train)]
val_df = df[~df['InvoiceNo'].isin(invoiceNoes_train)]

### create Sentance constructor

In [66]:
purchases_train = []
# use tqdm to show progress bar
for idx in tqdm(invoiceNoes_train): ## invoiceNoes_train is list
    purchase = train_df[train_df['InvoiceNo'] == idx]
    purchases_train.append(purchase['StockCode'].tolist())

100%|██████████| 16518/16518 [04:00<00:00, 68.54it/s]


## Create and train model

### create callback function

In [68]:
from IPython.display import clear_output

# create class for monitoring training
class MonitorCallback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
        clear_output(wait=True)
        print("Training started")

    def on_epoch_begin(self, model):
        pass
        # print(f"Epoch {self.epoch} started")

    def on_epoch_end(self, model):
        clear_output(wait=True)
        print(f"Epoch {self.epoch+1}/{model.epochs}")
        self.epoch += 1

monitor = MonitorCallback()

Training started


### create and train model

In [70]:
model = Word2Vec(sentences=purchases_train, 
                 vector_size=50, 
                 window=5, 
                 workers=4, 
                 epochs=500, 
                 callbacks=[monitor])

Epoch 540/500


## Check model

In [71]:
print(model)

Word2Vec<vocab=3434, vector_size=50, alpha=0.025>


In [72]:
print(model.wv.index_to_key[:10])

['85123A', '85099B', '22423', '47566', '20725', '84879', '22720', '22197', '21212', '22383']


In [73]:
model.wv['22423']

array([-1.8187104 , -1.1681801 , -0.13318571,  0.3091863 ,  2.0023158 ,
        3.1071742 ,  1.1892548 ,  2.7869477 , -1.6248521 , -2.3747544 ,
       -0.3482558 , -0.8136302 , -0.8610946 ,  0.07191494, -2.7545547 ,
        0.6575812 ,  2.1034913 ,  0.7181856 ,  0.62349474,  1.1743102 ,
        1.6093544 , -0.01631608,  1.4018252 , -1.4080409 ,  7.3188434 ,
       -0.5622538 ,  2.4437628 , -1.8096857 , -0.46645996,  0.9483575 ,
       -2.5570695 ,  1.010383  , -3.2150342 , -0.23169959,  0.06326795,
        2.9140418 ,  2.482141  , -4.2545776 ,  3.5602882 ,  2.2424054 ,
       -1.8977127 , -0.4605016 , -1.088804  ,  0.09825351,  1.2692664 ,
        2.533066  ,  2.5236895 ,  1.2904867 ,  0.7226628 ,  3.2321181 ],
      dtype=float32)

## Save model

In [74]:
model_save_path = "../models/online_retail.model"
if os.path.exists(os.path.dirname(model_save_path)):  
    if os.path.exists(model_save_path):
        print(f"Model already exists at {model_save_path}. Overwriting...")
    else:  
        model.save("../models/online_retail.model")
else:
    print(f"Directory {os.path.dirname(model_save_path)} does not exist. Please check the path.")

del model_save_path

## Use model

In [ ]:
similar_product = model.wv.most_similar('22423', topn=5)
print(train_df[train_df['StockCode'] == '22423']['Description'].values[0])
print("Similar products to '22423':")
for product, score in similar_product:
    print(f"Product: {product}, Similarity Score: {score:.4f}")
    print(train_df[train_df['StockCode'] == product]['Description'].values[0])
    print("-" * 50)
del similar_product

REGENCY CAKESTAND 3 TIER
Similar products to '22423':
Product: 22427, Similarity Score: 0.5962
ENAMEL FLOWER JUG CREAM
--------------------------------------------------
Product: 22424, Similarity Score: 0.5580
ENAMEL BREAD BIN CREAM
--------------------------------------------------
Product: 22697, Similarity Score: 0.5115
GREEN REGENCY TEACUP AND SAUCER
--------------------------------------------------
Product: 22426, Similarity Score: 0.4933
ENAMEL WASH BOWL CREAM
--------------------------------------------------
Product: 23173, Similarity Score: 0.4921
REGENCY TEAPOT ROSES 
--------------------------------------------------


In [102]:
products = train_df[['StockCode', 'Description']]

print(products.head(5))
# remove duplicates
products = products.drop_duplicates(subset='StockCode',
                                    keep='last',
                                    )
print(products.head(5))

  StockCode                          Description
0    85123A   WHITE HANGING HEART T-LIGHT HOLDER
1     71053                  WHITE METAL LANTERN
2    84406B       CREAM CUPID HEARTS COAT HANGER
3    84029G  KNITTED UNION FLAG HOT WATER BOTTLE
4    84029E       RED WOOLLY HOTTIE WHITE HEART.
     StockCode                          Description
107      84854                  GIRLY PINK TOOL SET
2313     82615  PINK MARSHMALLOW SCARF KNITTING KIT
4116    35271S                 GOLD PRINT PAPER BAG
4582     21268             VINTAGE BLUE TINSEL REEL
5024    16161M                     WRAP  PINK FLOCK


In [108]:
products_dict = products.groupby('StockCode')['Description'].apply(list).to_dict()

In [109]:
sku_now = '15044C'
similar = model.wv.most_similar(sku_now, topn=7)

print('Shopping:', sku_now, products_dict[sku_now][0])
print('-'*50)

for i in similar:
    if i[1] > 0.6:
        print(f'{i[0]:6} {i[1]:.4f} {products_dict[i[0]][0]}')

Shopping: 15044C PURPLE PAPER PARASOL
--------------------------------------------------
15044B 0.7527 BLUE PAPER PARASOL 
15044D 0.6889 RED PAPER PARASOL
15044A 0.6652 PINK PAPER PARASOL 


## Plotting follow in book